In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Azure Blob Storage

In [ ]:
search_url = os.getenv("SEARCH_SERVICE_URL")
search_api_key = os.getenv("SEARCH_SERVICE_API_KEY")
blob_connection_string = os.getenv("STORAGE_CONNECTION_STRING")
blob_container_name = os.getenv("STORAGE_CONTAINER_NAME")
foundry_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
foundry_model_api_key = os.getenv("FOUNDRY_MODEL_API_KEY")
azure_openai_endpoint = (
    os.getenv("AZURE_OPENAI_ENDPOINT")
    or os.getenv("AZURE_OPENAI_RESOURCE_URL")
    or os.getenv("AZURE_OPENAI_ENDPOINT_URL")
    or "https://foundry-rag-nagh.cognitiveservices.azure.com/"
)
azure_openai_api_key = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_KEY")
    or foundry_model_api_key
)
llm_model_name = os.getenv("LLM_MODEL_NAME")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
print("Env vars loaded.")

In [19]:
# Create a blob knowledge source
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import AzureBlobKnowledgeSource, AzureBlobKnowledgeSourceParameters, KnowledgeBaseAzureOpenAIModel, AzureOpenAIVectorizerParameters, KnowledgeSourceContentExtractionMode
from azure.search.documents.knowledgebases.models import KnowledgeSourceIngestionParameters, KnowledgeSourceAzureOpenAIVectorizer

index_client = SearchIndexClient(endpoint=search_url, credential=AzureKeyCredential(search_api_key))

if not azure_openai_endpoint or not azure_openai_api_key:
    raise ValueError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file. The Foundry project endpoint is not the correct vectorization target for Azure AI Search knowledge sources.")

# If a KB still references a source, Azure Search blocks deleting the source. Delete the KB first,
# then recreate the sources with the current embedding settings.
for kb in index_client.list_knowledge_bases():
    if kb.name == "health-banking-kb":
        print(f"Deleting existing knowledge base '{kb.name}' before source recreation.")
        index_client.delete_knowledge_base(kb.name)

# Existing knowledge sources cannot change embedding model configuration. If a source with the same
# name already exists from a previous run, delete it before recreating it with the new vectorizer.
existing_blob_ks = [ks for ks in index_client.list_knowledge_sources() if ks.name == "my-blob-ks-2"]
if existing_blob_ks:
    index_client.delete_knowledge_source("my-blob-ks-2")

knowledge_source = AzureBlobKnowledgeSource(
    name="my-blob-ks-2",
    description="This knowledge source pulls from a blob storage container.",
    encryption_key=None,
    azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
        connection_string=blob_connection_string,
        container_name=blob_container_name,
        folder_path=None,
        is_adls_gen2=False,
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            identity=None,
            disable_image_verbalization=False,
            chat_completion_model=KnowledgeBaseAzureOpenAIModel(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint.rstrip("/"),
                    deployment_name=llm_model_name,
                    model_name=llm_model_name,
                    api_key=azure_openai_api_key,
                )
            ),
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint.rstrip("/"),
                    deployment_name=embedding_model_name,
                    model_name=embedding_model_name,
                    api_key=azure_openai_api_key,
                )
            ),
            content_extraction_mode=KnowledgeSourceContentExtractionMode.MINIMAL,
            ingestion_schedule=None,
        )
    )
)

index_client.create_or_update_knowledge_source(knowledge_source)
print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")

Knowledge source 'my-blob-ks-2' created or updated successfully.


In [7]:
# Check knowledge source ingestion status
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
import json


status = index_client.get_knowledge_source_status("my-blob-ks-2")
print(json.dumps(status.as_dict(), indent=2))

{
  "@odata.context": "https://searchservice-nagh.search.windows.net/$metadata#Microsoft.WindowsAzure.Search.Core.Models.V2026_05_01_Preview.KnowledgeSources.KnowledgeSourceStatus",
  "kind": "azureBlob",
  "synchronizationStatus": "creating",
  "synchronizationInterval": null,
  "currentSynchronizationState": null,
  "lastSynchronizationState": null,
  "statistics": null
}


### Azure SQL

In [14]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    ContentColumnMapping,
    EmbeddingColumnMapping,
    IndexedSqlKnowledgeSource,
    IndexedSqlKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

In [11]:
TABLE_CONFIGS = {
    "fdic_institutions": {
        "description": "FDIC bank institutions — name, state, assets, deposits",
        "content_cols": [("NAME", "Edm.String"), ("STALP", "Edm.String")],
        "embed_col": "NAME",
    },
    "fdic_locations": {
        "description": "FDIC bank branch locations — name, city, state, ZIP",
        "content_cols": [
            ("NAME", "Edm.String"),
            ("CITY", "Edm.String"),
            ("STALP", "Edm.String"),
            ("ZIP", "Edm.String"),
        ],
        "embed_col": "NAME",
    },
    "fdic_financials": {
        "description": "FDIC bank financial reports — name, report ID, ROA, ROE",
        "content_cols": [("NAME", "Edm.String"), ("ID", "Edm.String")],
        "embed_col": "NAME",
    },
    # FRED time-series — only OBSERVATION_DATE is varchar; numeric values are float
    # These are indexed by date; use the SQL service to query actual values.
    "fred_fedfunds":      {"description": "Federal Funds Rate time series",                   "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_dgs10":         {"description": "10-Year Treasury Constant Maturity Rate",          "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_cpi":           {"description": "Consumer Price Index (CPIAUCSL) time series",      "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_unemployment":  {"description": "Unemployment Rate (UNRATE) time series",           "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_gdp":           {"description": "US GDP time series",                               "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_bank_credit":   {"description": "Bank Credit of All Commercial Banks time series",  "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_mortgage":      {"description": "30-Year Fixed Rate Mortgage Average time series",  "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_delinquency":   {"description": "Delinquency Rate on Consumer Loans time series",   "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_health_spending":{"description": "Health Care Expenditures time series",            "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
    "fred_exchange":      {"description": "US/Euro Exchange Rate (DEXUSEU) time series",      "content_cols": [("OBSERVATION_DATE", "Edm.String")], "embed_col": "OBSERVATION_DATE"},
}



In [ ]:
sql_connection = os.getenv("SQL_CONNECTION_STRING")
print("SQL connection string loaded." if sql_connection else "ERROR: SQL_CONNECTION_STRING not set in .env")

In [8]:
def create_knowledge_source(
    client: SearchIndexClient,
    table_name: str,
    config: dict,
) -> None:
    if not azure_openai_endpoint or not azure_openai_api_key:
        raise ValueError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file before creating SQL knowledge sources.")

    source_name = f"ks-{table_name.replace('_', '-')}"
    existing_sources = [ks for ks in client.list_knowledge_sources() if ks.name == source_name]
    if existing_sources:
        print(f"Existing knowledge source '{source_name}' found; deleting it so embedding config can be recreated.")
        client.delete_knowledge_source(source_name)

    embedding_params = AzureOpenAIVectorizerParameters(
        resource_url=azure_openai_endpoint.rstrip("/"),
        deployment_name=embedding_model_name,
        model_name=embedding_model_name,
        api_key=azure_openai_api_key,
    )

    ingestion_params = KnowledgeSourceIngestionParameters(
        content_extraction_mode="minimal",
        embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
            azure_open_ai_parameters=embedding_params
        ),
    )

    knowledge_source = IndexedSqlKnowledgeSource(
        name=source_name,
        description=config["description"],
        indexed_sql_parameters=IndexedSqlKnowledgeSourceParameters(
            connection_string=sql_connection,
            table_or_view=f"dbo.{table_name}",
            content_columns=[
                ContentColumnMapping(
                    name=col_name.lower(),
                    source_field=col_name,
                    search_field_type=edm_type,
                )
                for col_name, edm_type in config["content_cols"]
            ],
            embedding_columns=[
                EmbeddingColumnMapping(
                    name=f"{config['embed_col'].lower()}_vector",
                    source_field=config["embed_col"],
                )
            ],
            ingestion_parameters=ingestion_params,
        ),
    )

    client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"  OK  {source_name}")


In [17]:
# Remove any stale knowledge base that still references sources before deleting/recreating them.
for kb in index_client.list_knowledge_bases():
    if kb.name == "health-banking-kb":
        print(f"Deleting existing knowledge base '{kb.name}' before rebuilding sources.")
        index_client.delete_knowledge_base(kb.name)

errors = []
for table_name, config in TABLE_CONFIGS.items():
    try:
        create_knowledge_source(index_client, table_name, config)
    except Exception as e:
        print(f"  FAIL  ks-{table_name}: {e}")
        errors.append(table_name)

if errors:
    print(f"Failed tables: {errors}")

print(f"\nDone. {len(TABLE_CONFIGS) - len(errors)} created, {len(errors)} failed.")

  OK  ks-fdic-institutions
  OK  ks-fdic-locations
  OK  ks-fdic-financials
  OK  ks-fred-fedfunds
  OK  ks-fred-dgs10
  OK  ks-fred-cpi
  OK  ks-fred-unemployment
  OK  ks-fred-gdp
  OK  ks-fred-bank-credit
  OK  ks-fred-mortgage
  OK  ks-fred-delinquency
  OK  ks-fred-health-spending
  OK  ks-fred-exchange

Done. 13 created, 0 failed.


In [ ]:

for ks in index_client.list_knowledge_sources():
    print(f"  - {ks.name} ({ks.kind})")

  - ks-azureblob-829 (azureBlob)
  - ks-rag-pdf-docs (azureBlob)
  - my-blob-ks (azureBlob)
  - my-blob-ks-2 (azureBlob)
